In [1]:
## IMPORT

import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf           # for OLS (smf.ols)
from sklearn.metrics import mean_squared_error  # perform mean-squared for RMSE
from statsmodels.stats.anova import anova_lm    # perform Chi-Squared test

## IGNORE WARNINGS

warnings.filterwarnings("ignore")

## LOAD DATA

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_PCA_for_test.csv")
# test = pd.read_csv("data/GDP_Forecast_test.csv")

## DEFINE FUNCTIONS FOR REUSE

def calc_performance(formula1, model1, formula2, model2):
    print(pd.DataFrame({'model'   : [formula1, formula2],
                    'Adj.R^2' : [model1.rsquared_adj, model2.rsquared_adj]}))
    print()
    print(anova_lm(model1, model2, test = "Chisq"))

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

## CONVERT YEAR-QUARTER: SPLIT COLUMNS
# we are using a different dataset for test because we want to calculate RMSE before submitting

# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]

## USING SEPARATE TEST DATASET FOR TESTING RMSE

# convert the data to YQ, year and quarter
test["observation_date"] = pd.to_datetime(test["observation_date"], dayfirst = True)
test["year"] = test["observation_date"].dt.year
test["qtr"] = "Q" + test["observation_date"].dt.quarter.astype(str)
test["YQ"] = test["year"].astype(str) + test["qtr"]
# remove observation_date column and move GDP to the back while changing col name to NGDP
test.drop(columns = "observation_date", inplace = True)
col = test.pop("GDP_PCA")
test["NGDP"] = col
# get the quarterly
test.sort_values(by = "YQ")
# # check data for 1990 - 2015 is similar to train to continue with this dataset for test
# test[test["year"] <= 2015].transpose()
## NOTE: data from later years is not as accurate, but we will use this to tentatively test our RMSE
test = test[test["year"]>= 2016].reset_index(drop = True)

## LOAD DATA

compustat = pd.read_csv("data/compustat_quarterly_financials2.csv")
# dropping columns with no unique data + gvkey
compustat.drop(columns = ["costat", "curcdq", "datafmt", "indfmt", "consol", "gvkey", "datadate"], inplace = True)

## CREATING AGGREGATE TO GET QUARTERLY DATA

agg_dict = {"revtq": "sum", "invtq": "sum", "oibdpq": "sum", "ppentq": "sum", "xrdq": "sum"}
compustat_macro = compustat.groupby("datacqtr").agg(agg_dict).reset_index()
compustat_macro.sort_values(by = "datacqtr", ascending = True, inplace = True) # for shift

## CREATING GROWTH AND GROWTH LAG

for col in agg_dict.keys():
    # calculating respective lags
    compustat_macro[f"{col}_lag"] = compustat_macro[col].shift(1)
    # calculating respective growth rates
    compustat_macro[f"{col}_growth"] = compustat_macro[col].pct_change()
    # creating lags for forecasting
    compustat_macro[f"{col}_growth_lag"] = compustat_macro[f"{col}_growth"].shift(1)
# convert infinite values to NaN for .drop()
compustat_macro.replace([np.inf, -np.inf], np.nan, inplace = True)
# drop missing values
compustat_macro.dropna(inplace = True)

## MERGING COMPUSTAT DATA TO TRAIN AND TEST

train_comp_macro = train.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")

### INTERNAL TESTING OF SPF_DATA

In [14]:
## LOAD DATA

BAABOND = pd.read_excel("data/spf_data/Mean_BAABOND_Level.xlsx")
BOND = pd.read_excel("data/spf_data/Mean_BOND_Level.xlsx")
EMP = pd.read_excel("data/spf_data/Mean_EMP_Level.xlsx")
UNEMP = pd.read_excel("data/spf_data/Mean_UNEMP_Level.xlsx")
CPROF = pd.read_excel("data/spf_data/Mean_CPROF_Level.xlsx")
HOUSING = pd.read_excel("data/spf_data/Mean_HOUSING_Level.xlsx")
INDPROD = pd.read_excel("data/spf_data/Mean_INDPROD_Level.xlsx")
TBILL = pd.read_excel("data/spf_data/Mean_TBILL_Level.xlsx")
TBOND = pd.read_excel("data/spf_data/Mean_TBOND_Level.xlsx")
NGDP = pd.read_excel("data/spf_data/Mean_NGDP_Level.xlsx")
PGDP = pd.read_excel("data/spf_data/Mean_PGDP_Level.xlsx")

In [15]:
## CONDUCTING CLEANING

# BAABOND
# NOTE: VALUES ONLY START AT 2010
BAABOND_ = BAABOND.copy()
BAABOND_ = BAABOND_[(BAABOND_["YEAR"] >= 1990) & (BAABOND_["YEAR"] <= 2020)]
BAABOND_["YQ"] = BAABOND_["YEAR"].astype(str) + "Q" + BAABOND_["QUARTER"].astype(str)
# drop columns that are empty
BAABOND_.drop(columns = ["BAABOND1"], inplace = True)

# BOND
BOND_ = BOND.copy()
BOND_ = BOND_[(BOND_["YEAR"] >= 1990) & (BOND_["YEAR"] <= 2020)]
BOND_["YQ"] = BOND_["YEAR"].astype(str) + "Q" + BOND_["QUARTER"].astype(str)
# drop columns that are empty
BOND_.drop(columns = ["BOND1"], inplace = True)

# EMP
# NOTE: VALUES ONLY START AT 2003
EMP_ = EMP.copy()
EMP_ = EMP_[(EMP_["YEAR"] >= 1990) & (EMP_["YEAR"] <= 2020)]
EMP_["YQ"] = EMP_["YEAR"].astype(str) + "Q" + EMP_["QUARTER"].astype(str)

# UNEMP
# NOTE: UNEMPC AND UNEMPD ONLY STARTS 2009
UNEMP_ = UNEMP.copy()
UNEMP_ = UNEMP_[(UNEMP_["YEAR"] >= 1990) & (UNEMP_["YEAR"] <= 2020)]
UNEMP_["YQ"] = UNEMP_["YEAR"].astype(str) + "Q" + UNEMP_["QUARTER"].astype(str)

# CPROF
CPROF_ = CPROF.copy()
CPROF_ = CPROF_[(CPROF_["YEAR"] >= 1990) & (CPROF_["YEAR"] <= 2020)]
CPROF_["YQ"] = CPROF_["YEAR"].astype(str) + "Q" + CPROF_["QUARTER"].astype(str)

# HOUSING
HOUSING_ = HOUSING.copy()
HOUSING_ = HOUSING_[(HOUSING_["YEAR"] >= 1990) & (HOUSING_["YEAR"] <= 2020)]
HOUSING_["YQ"] = HOUSING_["YEAR"].astype(str) + "Q" + HOUSING_["QUARTER"].astype(str)

# INDPROD
INDPROD_ = INDPROD.copy()
INDPROD_ = INDPROD_[(INDPROD_["YEAR"] >= 1990) & (INDPROD_["YEAR"] <= 2020)]
INDPROD_["YQ"] = INDPROD_["YEAR"].astype(str) + "Q" + INDPROD_["QUARTER"].astype(str)

# TBILL
# NOTE: TBILLC AND TBILLD ONLY STARTS 2009
TBILL_ = TBILL.copy()
TBILL_ = TBILL_[(TBILL_["YEAR"] >= 1990) & (TBILL_["YEAR"] <= 2020)]
TBILL_["YQ"] = TBILL_["YEAR"].astype(str) + "Q" + TBILL_["QUARTER"].astype(str)

# TBOND
# NOTE: TBOND1 - TBONDB ONLY STARTS 1992
# NOTE: TBONDC AND TBONDD ONLY STARTS 2009
TBOND_ = TBOND.copy()
TBOND_ = TBOND_[(TBOND_["YEAR"] >= 1990) & (TBOND_["YEAR"] <= 2020)]
TBOND_["YQ"] = TBOND_["YEAR"].astype(str) + "Q" + TBOND_["QUARTER"].astype(str)

# NGDP
NGDP_ = NGDP.copy()
NGDP_ = NGDP_[(NGDP_["YEAR"] >= 1990) & (NGDP_["YEAR"] <= 2020)]
NGDP_["YQ"] = NGDP_["YEAR"].astype(str) + "Q" + NGDP_["QUARTER"].astype(str)

# PGDP
PGDP_ = PGDP.copy()
PGDP_ = PGDP_[(PGDP_["YEAR"] >= 1990) & (PGDP_["YEAR"] <= 2020)]
PGDP_["YQ"] = PGDP_["YEAR"].astype(str) + "Q" + PGDP_["QUARTER"].astype(str)

In [ ]:
BAABOND_spf = BAABOND_[["YQ", "BAABOND2", "BAABOND3"]]
BOND_spf = BOND_[["YQ", "BOND2", "BOND3"]]
EMP_spf = EMP_[["YQ", "EMP2", "EMP3"]]
UNEMP_spf = UNEMP_[["YQ", "UNEMP2", "UNEMP3"]]
CPROF_spf = CPROF_[["YQ", "CPROF2", "CPROF3"]]
HOUSING_spf = HOUSING_[["YQ", "HOUSING2", "HOUSING3"]]
INDPROD_spf = INDPROD_[["YQ", "INDPROD2", "INDPROD3"]]
TBILL_spf = TBILL_[["YQ", "TBILL2", "TBILL3"]]
TBOND_spf = TBOND_[["YQ", "TBOND2", "TBOND3"]]
NGDP_spf = NGDP_[["YQ", "NGDP2", "NGDP3"]]
PGDP_spf = PGDP_[["YQ", "PGDP2", "PGDP3"]]

,YQ,PGDP2,PGDP3
85,1990Q1,129.3462,130.6000
86,1990Q2,130.9000,132.2333
87,1990Q3,132.3077,133.7000
88,1990Q4,133.6667,135.1933
89,1991Q1,134.5771,135.8029
...,...,...,...
204,2019Q4,113.1465,113.6922
205,2020Q1,113.5831,114.1358
206,2020Q2,113.3699,113.6600
207,2020Q3,113.3225,113.7419


In [ ]:
BAABOND_spf
BOND_spf
EMP_spf
UNEMP_spf
CPROF_spf
HOUSING_spf
INDPROD_spf
TBILL_spf
TBOND_spf
NGDP_spf
PGDP_spf

In [6]:
## IMPORT

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV

## STORING X AND Y FOR TRAIN AND TEST

features = [x for x in train_comp_macro.columns.tolist() if "lag" in x]

X_train = train_comp_macro[features]
y_train = train_comp_macro["NGDP"]
X_test = test_comp_macro[features]
y_test = test_comp_macro["NGDP"]

## SCALING BEFORE REGULARISATION (FEATURES ARE ON DIFFERENT SCALES)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)        # fit on train only, transform test

## FITTING LASSO MODEL ON TRAINING SET

lasso = LassoCV(cv = 5)
lasso.fit(X_train_scaled, y_train)
rmse_lasso = np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_scaled)))
print(f"Lasso   | RMSE: {rmse_lasso:.4f}    | alpha: {lasso.alpha_}")

## FITTING RIDGE MODEL ON TRAINING SET

ridge = RidgeCV(cv = 5)
ridge.fit(X_train_scaled, y_train)
rmse_ridge = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_scaled)))
print(f"Ridge   | RMSE: {rmse_ridge:.4f}    | alpha: {ridge.alpha_}")

## FITTING ELASTICNET MODEL ON TRAINING SET

enet = ElasticNetCV(l1_ratio = [0.1, 0.5, 0.9], cv = 5)
enet.fit(X_train_scaled, y_train)
rmse_enet = np.sqrt(mean_squared_error(y_test, enet.predict(X_test_scaled)))
print(f"ElasNet | RMSE: {rmse_enet:.4f}    | alpha: {enet.alpha_}   | l1_ratio: {enet.l1_ratio_}")

## COMPARING FEATURE WEIGHTS ACROSS MODELS

print("\nFeature coefficients:")
print(f"{'Feature':<25} {'Lasso':>10} {'Ridge':>10} {'ElasNet':>10}")
for feat, lc, rc, ec in zip(features, lasso.coef_, ridge.coef_, enet.coef_):
    print(f"{feat:<25} {lc:>10.4f} {rc:>10.4f} {ec:>10.4f}")

Lasso   | RMSE: 11.1985    | alpha: 0.25593564914553335
Ridge   | RMSE: 11.2110    | alpha: 10.0
ElasNet | RMSE: 11.1728    | alpha: 0.9635812897026922   | l1_ratio: 0.1

Feature coefficients:
Feature                        Lasso      Ridge    ElasNet
revtq_lag                    -0.0000    -0.1967    -0.1144
revtq_growth_lag              0.0000     0.0346     0.0000
invtq_lag                    -0.0000    -0.3487    -0.1025
invtq_growth_lag              0.0000     0.3519     0.0059
oibdpq_lag                   -0.0000     0.8499    -0.0000
oibdpq_growth_lag             0.0000     0.0977     0.0000
ppentq_lag                   -0.3202    -0.7450    -0.1948
ppentq_growth_lag             0.0000    -0.0603     0.0000
xrdq_lag                     -0.3182    -0.4097    -0.2157
xrdq_growth_lag               0.0000    -0.3980    -0.0000
